# SASRec — MovieLens 1M · Full-Rank Evaluation

Pipeline chuyển đổi từ BERT4Rec sang SASRec.  
Đánh giá theo **Full-Rank** (rank target trong toàn bộ item set).

In [1]:
import os, math, random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sasrec import SASRec          # ← file sasrec.py (đã gộp)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Torch version: 2.5.1+cu121
CUDA available: True
Device: cuda


## 1. Đọc dữ liệu

In [2]:
RATINGS_PATH = r'F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\ratings.dat'
# Nếu chạy trên Linux / Colab, đổi thành path tương ứng, ví dụ:
# RATINGS_PATH = './data/ml-1m/ratings.dat'

def load_data(ratings_file):
    ratings = pd.read_csv(
        ratings_file,
        sep="::",
        engine="python",
        names=["user", "item", "rating", "timestamp"]
    )
    print(f"Loaded {len(ratings):,} interactions.")
    return ratings

ratings = load_data(RATINGS_PATH)
ratings.head()


Loaded 1,000,209 interactions.


,user,item,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


## 2. K-Core Filtering (k = 3)

In [3]:
def k_core_filter(df, k=3):
    while True:
        n_before = len(df)
        user_cnt  = df["user"].value_counts()
        df = df[df["user"].isin(user_cnt[user_cnt >= k].index)]
        item_cnt  = df["item"].value_counts()
        df = df[df["item"].isin(item_cnt[item_cnt >= k].index)]
        if len(df) == n_before:
            break
    return df

ratings = ratings.sort_values(["user", "timestamp"]).reset_index(drop=True)
ratings = k_core_filter(ratings, k=3)

print(f"Users : {ratings['user'].nunique():,}")
print(f"Items : {ratings['item'].nunique():,}")
print(f"Interactions: {len(ratings):,}")


Users : 6,040
Items : 3,503
Interactions: 999,917


## 3. Re-index items (bắt đầu từ 1)

In [4]:
def reindex_items(df):
    unique_items = sorted(df["item"].unique())
    movie2id = {raw: idx + 1 for idx, raw in enumerate(unique_items)}
    id2movie  = {v: k for k, v in movie2id.items()}
    df = df.copy()
    df["item"] = df["item"].map(movie2id)
    return df, movie2id, id2movie, len(movie2id)

ratings, movie2id, id2movie, num_items = reindex_items(ratings)
print(f"num_items = {num_items}  (item IDs: 1 … {num_items})")


num_items = 3503  (item IDs: 1 … 3503)


## 4. Xây dựng chuỗi tương tác

In [5]:
user_sequences = (
    ratings.groupby("user")["item"]
    .apply(list)
    .to_dict()
)
print(f"Total users with sequences: {len(user_sequences):,}")

seq_lengths = [len(s) for s in user_sequences.values()]
print(f"Seq length — min: {min(seq_lengths)}, max: {max(seq_lengths)}, "
      f"median: {int(np.median(seq_lengths))}")


Total users with sequences: 6,040
Seq length — min: 19, max: 2290, median: 96


## 5. Chia dữ liệu — Leave-One-Out

In [6]:
MAX_LEN = 200

train_data = []   # list of (input_seq [T], pos_seq [T])  — sliding window
val_seqs   = []   # list of (input_seq, target_item)
test_seqs  = []   # list of (input_seq, target_item)

def pad_or_truncate(seq, max_len):
    """Left-pad with 0 if shorter, truncate left if longer."""
    if len(seq) >= max_len:
        return seq[-max_len:]
    return [0] * (max_len - len(seq)) + seq

for user, seq in user_sequences.items():
    if len(seq) < 3:          # cần ít nhất 3 items
        continue

    test_seqs.append((seq[:-1],  seq[-1]))
    val_seqs .append((seq[:-2],  seq[-2]))

    train_raw = seq[:-2]
    for end in range(2, len(train_raw) + 1):
        sub = train_raw[:end]
        inp = pad_or_truncate(sub[:-1], MAX_LEN)   # input
        pos = pad_or_truncate(sub[1:],  MAX_LEN)   # shifted positive
        train_data.append((inp, pos))

print(f"Train samples  : {len(train_data):,}")
print(f"Val  users     : {len(val_seqs):,}")
print(f"Test users     : {len(test_seqs):,}")


Train samples  : 981,797
Val  users     : 6,040
Test users     : 6,040


## 6. Dataset & DataLoader

In [7]:
class SASRecTrainDataset(Dataset):
    """
    Mỗi sample = (input_seq [T], pos_seq [T], neg_seq [T]).
    Negative được sample ngẫu nhiên tại __getitem__ để đa dạng qua các epoch.
    """
    def __init__(self, data, num_items, max_len=200):
        self.data      = data          # list of (inp, pos)
        self.num_items = num_items
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        inp, pos = self.data[idx]
        neg = []
        pos_set = set(pos)
        while len(neg) < self.max_len:
            n = random.randint(1, self.num_items)
            if n not in pos_set:
                neg.append(n)
        return (
            torch.LongTensor(inp),
            torch.LongTensor(pos),
            torch.LongTensor(neg),
        )


def build_eval_loader(seqs, max_len, batch_size=256):
    """Tạo DataLoader cho val / test (input_seq + target)."""
    X, Y = [], []
    for input_seq, target in seqs:
        x = pad_or_truncate(input_seq, max_len)
        X.append(x)
        Y.append(target)
    ds = TensorDataset(torch.LongTensor(X), torch.LongTensor(Y))
    return DataLoader(ds, batch_size=batch_size, shuffle=False, pin_memory=True)


BATCH_SIZE = 256

train_dataset = SASRecTrainDataset(train_data, num_items, max_len=MAX_LEN)
train_loader  = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True
)

val_loader  = build_eval_loader(val_seqs,  MAX_LEN)
test_loader = build_eval_loader(test_seqs, MAX_LEN)

print(f"Train batches : {len(train_loader):,}")
print(f"Val  users    : {len(val_seqs):,}")
print(f"Test users    : {len(test_seqs):,}")


Train batches : 3,836
Val  users    : 6,040
Test users    : 6,040


## 7. Hyperparameters & Khởi tạo mô hình

In [8]:
import argparse

args = argparse.Namespace(
    maxlen       = MAX_LEN,
    hidden_units = 50,     # embedding / attention dim
    num_blocks   = 2,      # số Transformer block
    num_heads    = 1,      # số attention head
    dropout_rate = 0.2,
    l2_emb       = 0.0,    # weight decay cho embedding (dùng trong optimizer)
    lr           = 1e-3,
)

EPOCHS          = 200
EVAL_EVERY      = 10       # evaluate mỗi N epoch
PATIENCE        = 5        # early-stopping (tính theo lần eval)
MODEL_SAVE_PATH = "saved_models/sasrec_best.pth"

model = SASRec(
    usernum = ratings["user"].nunique(),
    itemnum = num_items,
    args    = args,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=args.lr,
    betas=(0.9, 0.98),
    weight_decay=args.l2_emb,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params : {n_params:,}")
print(f"item_emb shape   : {model.item_emb.weight.shape}")


Trainable params : 211,200
item_emb shape   : torch.Size([3504, 50])


## 8. Hàm đánh giá — Full-Rank

Target item được rank trong **toàn bộ item set** (không sample negative).  
Các chỉ số: **HR@K**, **NDCG@K**, **MRR** với K ∈ {1, 5, 10}.

In [9]:
@torch.no_grad()
def evaluate_fullrank(model, loader, num_items, device, ks=(1, 5, 10)):
    """
    Full-Rank evaluation.

    Với mỗi user:
      1. Lấy biểu diễn tại vị trí cuối cùng của chuỗi.
      2. Tính logit với TẤT CẢ num_items items (item IDs 1 … num_items).
      3. Rank target item trong danh sách đầy đủ đó.

    Args
    ----
    model      : SASRec instance (eval mode khi vào, train mode khi ra)
    loader     : DataLoader trả về (input_seq [B,T], target [B])
    num_items  : tổng số items (không tính padding 0)
    device     : torch device
    ks         : tuple of K values for HR@K, NDCG@K

    Returns
    -------
    metrics : dict  {"HR@k", "NDCG@k", "MRR"}
    total   : int   tổng số users đánh giá
    """
    model.eval()

    # Tất cả item IDs hợp lệ: 1 … num_items
    all_item_ids = torch.arange(1, num_items + 1, device=device)   # (num_items,)

    metrics = {f"HR@{k}":   0.0 for k in ks}
    metrics.update({f"NDCG@{k}": 0.0 for k in ks})
    metrics["MRR"] = 0.0
    total = 0

    for input_seq, targets in tqdm(loader, desc="Evaluating", leave=False):
        input_seq = input_seq.to(device)   # (B, T)
        targets   = targets.to(device)     # (B,)
        B         = input_seq.size(0)

        # ── Encode & lấy vector cuối ────────────────────────────────────
        seq_emb = model._encode(input_seq)         # (B, T, C)
        final   = seq_emb[:, -1, :]                # (B, C)

        # ── Score toàn bộ item set ───────────────────────────────────────
        all_emb = model.item_emb(all_item_ids)     # (num_items, C)
        logits  = final.matmul(all_emb.T)          # (B, num_items)

        # ── Tính rank của target item ────────────────────────────────────
        # all_item_ids[j] = j+1  →  index của target item = target - 1
        target_scores = logits[torch.arange(B), targets - 1]          # (B,)

        # Số items có score >= target_score (tính rank từ 1)
        ranks = (logits >= target_scores.unsqueeze(1)).sum(dim=1)      # (B,)

        for i in range(B):
            rank = ranks[i].item()
            metrics["MRR"] += 1.0 / rank
            for k in ks:
                if rank <= k:
                    metrics[f"HR@{k}"]   += 1.0
                    metrics[f"NDCG@{k}"] += 1.0 / math.log2(rank + 1)

        total += B

    if total > 0:
        for key in metrics:
            metrics[key] /= total

    model.train()
    return metrics, total


## 9. Huấn luyện

In [10]:
os.makedirs("saved_models", exist_ok=True)

best_val_hr10    = 0.0
patience_counter = 0
history          = []   # list of dict per epoch

print("=" * 65)
print(f"SASRec  |  {EPOCHS} epochs  |  device: {device}")
print(f"hidden_units={args.hidden_units}  num_blocks={args.num_blocks}  "
      f"num_heads={args.num_heads}  dropout={args.dropout_rate}")
print(f"EVAL_EVERY={EVAL_EVERY}  PATIENCE={PATIENCE}")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_auc  = 0.0

    pbar = tqdm(train_loader,
                desc=f"Epoch {epoch:03d}/{EPOCHS}",
                leave=True, dynamic_ncols=True)

    for inp_seq, pos_seq, neg_seq in pbar:
        inp_seq = inp_seq.to(device)
        pos_seq = pos_seq.to(device)
        neg_seq = neg_seq.to(device)

        optimizer.zero_grad()
        loss, auc = model(inp_seq, pos_seq, neg_seq)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        total_auc  += auc.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", auc=f"{auc.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    avg_auc  = total_auc  / len(train_loader)

    row = {"epoch": epoch, "train_loss": avg_loss, "train_auc": avg_auc}

    # ── Validation mỗi EVAL_EVERY epoch ────────────────────────────────
    if epoch % EVAL_EVERY == 0:
        val_metrics, n_val = evaluate_fullrank(
            model, val_loader, num_items, device, ks=(1, 5, 10)
        )
        val_hr10  = val_metrics["HR@10"]
        val_ndcg10 = val_metrics["NDCG@10"]
        val_mrr   = val_metrics["MRR"]
        row.update(val_metrics)

        print(f"\n  → [Val epoch {epoch}]"
              f"  HR@10={val_hr10:.4f}"
              f"  NDCG@10={val_ndcg10:.4f}"
              f"  MRR={val_mrr:.4f}"
              f"  (best HR@10={best_val_hr10:.4f})")

        if val_hr10 > best_val_hr10:
            best_val_hr10    = val_hr10
            patience_counter = 0
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"  → ✅ Saved best model! Val HR@10 = {val_hr10:.4f}")
        else:
            patience_counter += 1
            print(f"  → Không cải thiện [{patience_counter}/{PATIENCE}]")
            if patience_counter >= PATIENCE:
                print("!" * 65)
                print(f"EARLY STOPPING tại epoch {epoch} | "
                      f"Best Val HR@10: {best_val_hr10:.4f}")
                print("!" * 65)
                history.append(row)
                break

    history.append(row)

print("\n" + "=" * 65)
print(f"HOÀN THÀNH! Best Val HR@10 = {best_val_hr10:.4f}")
print(f"Model: {MODEL_SAVE_PATH}")
print("=" * 65)


SASRec  |  200 epochs  |  device: cuda
hidden_units=50  num_blocks=2  num_heads=1  dropout=0.2
EVAL_EVERY=10  PATIENCE=5


Epoch 001/200:   0%|          | 0/3836 [00:00<?, ?it/s]

Epoch 002/200:   0%|          | 0/3836 [00:00<?, ?it/s]

Epoch 003/200:   0%|          | 0/3836 [00:00<?, ?it/s]

Epoch 004/200:   0%|          | 0/3836 [00:00<?, ?it/s]

...

Epoch 091/200:   0%|          | 0/3836 [00:00<?, ?it/s]

Epoch 092/200:   0%|          | 0/3836 [00:00<?, ?it/s]

Epoch 093/200:   0%|          | 0/3836 [00:00<?, ?it/s]

Epoch 094/200:   0%|          | 0/3836 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 10. Đánh giá trên tập Test (Full-Rank)

In [ ]:
if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(
        torch.load(MODEL_SAVE_PATH, map_location=device, weights_only=True)
    )
    print(f"✅ Loaded best model: {MODEL_SAVE_PATH}")

KS = (1, 5, 10)

print("\n" + "=" * 55)
print("🎯 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST  (Full-Rank)")
print("=" * 55)
test_metrics, n_test = evaluate_fullrank(
    model, test_loader, num_items, device, ks=KS
)
for k_name, v in test_metrics.items():
    print(f"  {k_name:<12}: {v:.4f}")
print(f"  {'Users':<12}: {n_test:,}")
print("-" * 55)

print("\n📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP VAL  (Full-Rank)")
print("-" * 55)
val_metrics, n_val = evaluate_fullrank(
    model, val_loader, num_items, device, ks=KS
)
for k_name, v in val_metrics.items():
    print(f"  {k_name:<12}: {v:.4f}")
print(f"  {'Users':<12}: {n_val:,}")
print("-" * 55)


## 11. Biểu đồ Loss & Metrics

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="Train Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE Loss")
axes[0].set_title("Training Loss"); axes[0].legend(); axes[0].grid(True)

# AUC
axes[1].plot(hist_df["epoch"], hist_df["train_auc"], color="orange", label="Train AUC")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUC")
axes[1].set_title("Training AUC (approx.)"); axes[1].legend(); axes[1].grid(True)

# Val metrics (chỉ lấy các epoch có eval)
eval_df = hist_df.dropna(subset=["HR@10"])
if not eval_df.empty:
    axes[2].plot(eval_df["epoch"], eval_df["HR@10"],   label="HR@10",   marker="o")
    axes[2].plot(eval_df["epoch"], eval_df["NDCG@10"], label="NDCG@10", marker="s")
    axes[2].plot(eval_df["epoch"], eval_df["MRR"],     label="MRR",     marker="^")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Score")
    axes[2].set_title("Validation Metrics (Full-Rank)")
    axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig("sasrec_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: sasrec_training_curves.png")


## 12. Bảng tổng kết kết quả

In [ ]:
summary_rows = []
for split_name, metrics, n_users in [
    ("Val  (Full-Rank)", val_metrics,  n_val),
    ("Test (Full-Rank)", test_metrics, n_test),
]:
    row = {"Split": split_name, "Users": n_users}
    row.update({k: f"{v:.4f}" for k, v in metrics.items()})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Split")
print(summary_df.to_string())
